In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor

    xgb_available = True
except ImportError:
    xgb_available = False

# Settings
pd.set_option("display.max_columns", None)

In [3]:
# Load the raw dataset
maintenance = pd.read_csv("../data/fleet_maintenance_log.csv")

# Phase 3: Initial Data Inspection
print("=== Dataset Shape ===")
print(f"Rows: {maintenance.shape[0]}, Columns: {maintenance.shape[1]}\n")

print("=== Raw Info & Data Types ===")
print(maintenance.info())

print("\n=== Missing Values ===")
print(maintenance.isnull().sum())

print("\n=== Duplicate Records ===")
print(f"Total duplicates: {maintenance.duplicated().sum()}")

=== Dataset Shape ===
Rows: 282, Columns: 8

=== Raw Info & Data Types ===
<class 'pandas.DataFrame'>
RangeIndex: 282 entries, 0 to 281
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   event_date           282 non-null    str    
 1   vehicle_id           282 non-null    str    
 2   vehicle_type         282 non-null    str    
 3   odometer_km          282 non-null    int64  
 4   service_type         282 non-null    str    
 5   cost_usd             282 non-null    float64
 6   days_in_shop         282 non-null    int64  
 7   next_service_due_km  282 non-null    int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 27.6 KB
None

=== Missing Values ===
event_date             0
vehicle_id             0
vehicle_type           0
odometer_km            0
service_type           0
cost_usd               0
days_in_shop           0
next_service_due_km    0
dtype: int64

=== Duplicate Records ===
Tota

In [4]:
# =====================================================
# Explicit Data Type Conversion
# =====================================================

# 1. Dates
maintenance["event_date"] = pd.to_datetime(maintenance["event_date"])

# 2. Identifiers & Categories (Alphanumeric IDs match to String)
maintenance["vehicle_id"] = maintenance["vehicle_id"].astype("string")
maintenance["vehicle_type"] = maintenance["vehicle_type"].astype("category")
maintenance["service_type"] = maintenance["service_type"].astype("category")

# 3. Numeric Features
maintenance["odometer_km"] = pd.to_numeric(
    maintenance["odometer_km"], errors="coerce"
)
maintenance["cost_usd"] = pd.to_numeric(maintenance["cost_usd"], errors="coerce")
maintenance["days_in_shop"] = pd.to_numeric(
    maintenance["days_in_shop"], errors="coerce"
)

# 4. Target Variable
maintenance["next_service_due_km"] = pd.to_numeric(
    maintenance["next_service_due_km"], errors="coerce"
)

print("=== Post-Correction Data Types ===")
print(maintenance.dtypes)

=== Post-Correction Data Types ===
event_date             datetime64[us]
vehicle_id                     string
vehicle_type                 category
odometer_km                     int64
service_type                 category
cost_usd                      float64
days_in_shop                    int64
next_service_due_km             int64
dtype: object


In [5]:
# Sort chronologically by vehicle to ensure timeline validity
maintenance = maintenance.sort_values(by=["vehicle_id", "event_date"]).reset_index(
    drop=True
)

# Drop exact duplicates if any exist
maintenance = maintenance.drop_duplicates()

# Business Logic Filtering
initial_count = len(maintenance)

maintenance = maintenance[
    (maintenance["odometer_km"] >= 0)
    & (maintenance["cost_usd"] >= 0)
    & (maintenance["days_in_shop"] >= 0)
    & (
        maintenance["next_service_due_km"] > maintenance["odometer_km"]
    )  # Next service must be in the future
]

print(f"Removed {initial_count - len(maintenance)} records violating business logic rules.")

Removed 0 records violating business logic rules.


In [6]:
# Ensure data remains sorted for sequential calculations
maintenance = maintenance.sort_values(by=["vehicle_id", "event_date"]).reset_index(drop=True)

# =====================================================
# 1. Historical Lag & Cumulative Metrics (Per Vehicle)
# =====================================================

# Track how many times a vehicle has been serviced prior to this event
maintenance["maintenance_count"] = maintenance.groupby("vehicle_id").cumcount()

# Capture data from the immediate prior service
maintenance["previous_date"] = maintenance.groupby("vehicle_id")["event_date"].shift(1)
maintenance["previous_odometer"] = maintenance.groupby("vehicle_id")["odometer_km"].shift(1)
maintenance["previous_cost"] = maintenance.groupby("vehicle_id")["cost_usd"].shift(1)

# =====================================================
# 2. Time and Distance Deltas
# =====================================================

# Days elapsed since the last service
maintenance["days_since_last_service"] = (
    (maintenance["event_date"] - maintenance["previous_date"]).dt.days
)

# Distance driven since the last service
maintenance["distance_since_last_service"] = (
    maintenance["odometer_km"] - maintenance["previous_odometer"]
)

# Cumulative financial investment in the vehicle
maintenance["cumulative_cost"] = maintenance.groupby("vehicle_id")["cost_usd"].cumsum()

# Moving average cost over the last 3 service events
maintenance["rolling_cost_3"] = (
    maintenance.groupby("vehicle_id")["cost_usd"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

# Lifetime average distance traveled between service intervals
maintenance["avg_service_distance"] = (
    maintenance.groupby("vehicle_id")["distance_since_last_service"]
    .transform(lambda x: x.expanding(min_periods=1).mean())
)

# =====================================================
# 3. Calendar & Cyclical Temporal Features
# =====================================================
maintenance["year"] = maintenance["event_date"].dt.year
maintenance["month"] = maintenance["event_date"].dt.month
maintenance["quarter"] = maintenance["event_date"].dt.quarter
maintenance["day_of_week"] = maintenance["event_date"].dt.dayofweek

# Cyclical encoding for the month to capture seasonality (e.g., winter vs. summer wear)
maintenance["month_sin"] = np.sin(2 * np.pi * maintenance["month"] / 12)
maintenance["month_cos"] = np.cos(2 * np.pi * maintenance["month"] / 12)

# =====================================================
# 4. Handle First-Time Service Records
# =====================================================
# For a vehicle's very first recorded service, lag operations result in NaNs.
# We backfill these with 0 for counts/distances, and the current value for averages.
fill_values = {
    "days_since_last_service": 0,
    "distance_since_last_service": 0,
    "previous_cost": 0,
    "previous_odometer": 0
}
maintenance = maintenance.fillna(value=fill_values)

# For running averages where previous distance was 0, fill with the global vehicle type average if needed, or 0
maintenance["avg_service_distance"] = maintenance["avg_service_distance"].fillna(0)

print("=== Engineered Feature Matrix ===")
print(maintenance[[
    "vehicle_id", "event_date", "maintenance_count", 
    "days_since_last_service", "distance_since_last_service", "rolling_cost_3"
]].head(5))

=== Engineered Feature Matrix ===
  vehicle_id event_date  maintenance_count  days_since_last_service  \
0     VH-001 2022-03-29                  0                      0.0   
1     VH-001 2022-06-10                  1                     73.0   
2     VH-001 2022-07-13                  2                     33.0   
3     VH-001 2022-10-24                  3                    103.0   
4     VH-001 2022-12-28                  4                     65.0   

   distance_since_last_service  rolling_cost_3  
0                          0.0      138.870000  
1                       5710.0      107.480000  
2                       2855.0      110.866667  
3                       8565.0      115.820000  
4                       5139.0      126.230000  


In [7]:
# Check for any remaining nulls in our planned feature columns
features_to_check = [
    "odometer_km", "days_since_last_service", "distance_since_last_service",
    "maintenance_count", "cost_usd", "previous_cost", "rolling_cost_3",
    "cumulative_cost", "avg_service_distance", "days_in_shop",
    "year", "month", "quarter", "day_of_week", "month_sin", "month_cos"
]

print("=== Remaining Null Values in Final Feature List ===")
print(maintenance[features_to_check].isnull().sum())

=== Remaining Null Values in Final Feature List ===
odometer_km                    0
days_since_last_service        0
distance_since_last_service    0
maintenance_count              0
cost_usd                       0
previous_cost                  0
rolling_cost_3                 0
cumulative_cost                0
avg_service_distance           0
days_in_shop                   0
year                           0
month                          0
quarter                        0
day_of_week                    0
month_sin                      0
month_cos                      0
dtype: int64


In [8]:
# =====================================================
# Step 1 – One-Hot Encoding Categorical Variables
# =====================================================
categorical_cols = ["vehicle_type", "service_type"]

# Creating dummy variables, dropping first to avoid multicollinearity
maintenance_encoded = pd.get_dummies(
    maintenance, columns=categorical_cols, drop_first=True
)

# Ensure all boolean dummy columns are converted to integers (0 or 1) for ML models
dummy_cols = [
    col
    for col in maintenance_encoded.columns
    if any(cat in col for cat in categorical_cols)
]
for col in dummy_cols:
    maintenance_encoded[col] = maintenance_encoded[col].astype(int)

# =====================================================
# Step 2 – Define Feature Universes
# =====================================================
# Exclude unmappable IDs, target leakage variables, and raw dates
exclude_cols = ["vehicle_id", "date", "event_date", "previous_date", "next_service_due_km"]

# Base feature list containing all engineered metrics
base_features = [
    col for col in maintenance_encoded.columns if col not in exclude_cols
]

# Model A Feature Set (Includes Odometer)
X_with_odometer = maintenance_encoded[base_features]

# Model B Feature Set (Excludes Odometer - Testing history dependency)
X_without_odometer = X_with_odometer.drop(columns=["odometer_km"], errors="ignore")

# Define Target
y = maintenance_encoded["next_service_due_km"]

# =====================================================
# Step 3 – Chronological Split Function
# =====================================================
def split_dataset(X, y, split_ratio=0.80):
    split_idx = int(len(X) * split_ratio)

    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]

    return X_train, X_test, y_train, y_test


# Split both configurations
X_train_A, X_test_A, y_train, y_test = split_dataset(X_with_odometer, y)
X_train_B, X_test_B, _, _ = split_dataset(X_without_odometer, y)

# =====================================================
# Step 4 – Training & Evaluation Core Pipeline
# =====================================================
results = []


def evaluate_and_log(model_name, dataset_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    results.append(
        {
            "Dataset": dataset_name,
            "Model": model_name,
            "MAE": round(mae, 2),
            "RMSE": round(rmse, 2),
            "R²": round(r2, 4),
        }
    )


def execute_training(X_train, X_test, y_train, y_test, dataset_label):
    # Core Model Manifest
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }

    # Inject XGBoost dynamically if installed
    if xgb_available:
        models["XGBoost"] = XGBRegressor(
            random_state=42, n_estimators=300, learning_rate=0.05, max_depth=5
        )

    trained_instances = {}

    for name, model in models.items():
        # Fit model on respective training subset
        model.fit(X_train, y_train)

        # Generate test predictions
        preds = model.predict(X_test)

        # Log Metrics
        evaluate_and_log(name, dataset_label, y_test, preds)

        # Save instance for feature importance parsing
        trained_instances[name] = model

    return trained_instances


# Execute training runs
print("Training Model Set A (With Odometer)...")
models_A = execute_training(X_train_A, X_test_A, y_train, y_test, "With Odometer")

print("Training Model Set B (Without Odometer)...")
models_B = execute_training(X_train_B, X_test_B, y_train, y_test, "Without Odometer")

# =====================================================
# Step 5 – Compile Results Matrix
# =====================================================
results_df = pd.DataFrame(results).sort_values(by=["Dataset", "RMSE"])
print("\n" + "=" * 50 + "\n   MODEL PERFORMANCE LEADERBOARD\n" + "=" * 50)
print(results_df.to_string(index=False))

Training Model Set A (With Odometer)...
Training Model Set B (Without Odometer)...

   MODEL PERFORMANCE LEADERBOARD
         Dataset             Model      MAE     RMSE     R²
   With Odometer Gradient Boosting  2600.43  3859.96 0.9714
   With Odometer           XGBoost  3194.91  4864.37 0.9546
   With Odometer Linear Regression  5320.95  6888.15 0.9089
   With Odometer     Random Forest  4575.53  7354.11 0.8962
Without Odometer Gradient Boosting  4237.22  7363.01 0.8959
Without Odometer           XGBoost  4522.89  7377.20 0.8955
Without Odometer     Random Forest  6127.88  9031.95 0.8434
Without Odometer Linear Regression 17170.88 20859.25 0.1649


In [9]:
import pickle

# 1. Isolate the winning model instance
best_model = models_A["Gradient Boosting"]

# 2. Extract and match feature importances
importance_df = pd.DataFrame({
    "Feature": X_train_A.columns,
    "Importance": best_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("=== Top 10 Most Important Features ===")
print(importance_df.head(10))

# 3. Export the trained model for production/Streamlit deployment
with open("gradient_boosting_maintenance_model.pkl", "wb") as f:
    pickle.dump(best_model, f)  # Fixed here!
    
print("\nModel successfully exported as 'gradient_boosting_maintenance_model.pkl'")

=== Top 10 Most Important Features ===
                       Feature  Importance
0                  odometer_km    0.722925
4            previous_odometer    0.183780
1                     cost_usd    0.063190
21     service_type_Oil change    0.010821
2                 days_in_shop    0.004862
20  service_type_Major service    0.003320
3            maintenance_count    0.003064
10        avg_service_distance    0.002472
8              cumulative_cost    0.002051
9               rolling_cost_3    0.001089

Model successfully exported as 'gradient_boosting_maintenance_model.pkl'


In [10]:
# 1. Get the latest record for every single vehicle
latest_records = maintenance_encoded.sort_values("event_date").groupby("vehicle_id").last().reset_index()

# 2. Separate features using the identical structure Model A expects
X_latest = latest_records[X_with_odometer.columns]

# 3. Predict the exact odometer reading where their NEXT service is due
latest_records["predicted_next_service_km"] = best_model.predict(X_latest)

# 4. Calculate remaining kilometers until that service hits
latest_records["km_until_service"] = latest_records["predicted_next_service_km"] - latest_records["odometer_km"]

# 5. Classify urgency levels based on industry thresholds
def assign_priority(km_left):
    if km_left <= 1000:
        return "🔴 Immediate Service Required"
    elif km_left <= 3000:
        return "🟡 Service Due Soon"
    else:
        return "🟢 Low Priority"

latest_records["Maintenance_Status"] = latest_records["km_until_service"].apply(assign_priority)

print("\n=== Fleet Maintenance Breakdown ===")
print(latest_records["Maintenance_Status"].value_counts())


=== Fleet Maintenance Breakdown ===
Maintenance_Status
🟢 Low Priority    20
Name: count, dtype: int64


In [13]:
import joblib
from pathlib import Path

# Create models folder if it doesn't exist
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

# Save the best model
joblib.dump(best_model, model_dir / "maintenance_model.pkl")

# Save the feature list used to train the model
# Using X_with_odometer.columns to fix the NameError
joblib.dump(list(X_with_odometer.columns), model_dir / "maintenance_features.pkl")

print("✅ Maintenance model and features saved successfully!")

✅ Maintenance model and features saved successfully!


In [12]:
import plotly.express as px
import plotly.graph_objects as go

# =====================================================
# Chart 1: Fleet Maintenance Priority Distribution
# =====================================================
# Count occurrences to ensure all statuses display even if currently 0
status_counts = latest_records["Maintenance_Status"].value_counts().reset_index()
status_counts.columns = ["Status", "Count"]

# Define a strict professional color mapping matching our emojis
color_map = {
    "🔴 Immediate Service Required": "#EF553B",
    "🟡 Service Due Soon": "#FECB52",
    "🟢 Low Priority": "#00CC96"
}

fig_distribution = px.bar(
    status_counts,
    x="Count",
    y="Status",
    orientation="h",
    color="Status",
    color_discrete_map=color_map,
    title="Fleet Maintenance Urgency Status Breakdown",
    text_auto=True
)

fig_distribution.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_title="Number of Vehicles",
    yaxis_title="",
    height=350
)
fig_distribution.show()

# =====================================================
# Chart 2: Individual Vehicle Trajectory Dashboard
# =====================================================
# Sort vehicles by remaining mileage to see who is getting closest to a service
latest_records_sorted = latest_records.sort_values(by="km_until_service")

fig_trajectory = go.Figure()

# Add a bar showing current odometer reading
fig_trajectory.add_trace(go.Bar(
    x=latest_records_sorted["vehicle_id"],
    y=latest_records_sorted["odometer_km"],
    name="Current Odometer (km)",
    marker_color="#1f77b4"
))

# Add a stacked bar showing the remaining gap until predicted next service due
fig_trajectory.add_trace(go.Bar(
    x=latest_records_sorted["vehicle_id"],
    y=latest_records_sorted["km_until_service"],
    name="Kilometers Until Next Service",
    marker_color="#7f7f7f",
    opacity=0.6,
    hovertext=latest_records_sorted["Maintenance_Status"]
))

fig_trajectory.update_layout(
    barmode="stack",
    title="Vehicle Maintenance Horizons (Current vs. Predicted Service Mileage Target)",
    xaxis_title="Vehicle Identifier",
    yaxis_title="Odometer Reading / Distance (km)",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=550
)
fig_trajectory.show()